<a href="https://colab.research.google.com/github/Sayak-coder/HateSpeech-detection/blob/main/Subtask_(B)_Hate_Speech_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_en = pd.read_csv('/content/StereoQueerEval_EN_training.tsv',sep='\t')
df_en.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_EN_0001,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...,yes,yes_explicit,individual_l
1,training_EN_0002,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...,yes,yes_implicit,group_t
2,training_EN_0003,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall...",yes,yes_implicit,"group_l,g"
3,training_EN_0004,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...,yes,yes_implicit,group_l
4,training_EN_0005,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him,yes,yes_implicit,group_g


### EXperiment A
comment ony as input

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [5]:
df = df_en

In [6]:
df['hate_speech'].value_counts()

,count
hate_speech,
no,1407
yes_explicit,810
yes_implicit,772


In [7]:
df.isnull().sum()

,0
StereoQueerEval_id,0
yt_title,0
yt_description,0
yt_comment,0
stereotype,0
hate_speech,0
target,0


In [8]:
X = list(df['yt_comment'])
y = df['hate_speech'].map({'no':0, 'yes_explicit':1, 'yes_implicit':2}).tolist()

In [9]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)

In [10]:
!pip install transformers

### sentiment-roberta-large-english

In [11]:
from transformers import AutoTokenizer

MODEL = f"siebert/sentiment-roberta-large-english"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [12]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [13]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10
)

In [15]:
from transformers import AutoModelForSequenceClassification, Trainer

# Explicitly set num_labels=3 to match our targets (0, 1, 2) and ignore mismatched classification head sizes
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True
).to(device)

trainer_RoBERTa = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer_RoBERTa.train()

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: siebert/sentiment-roberta-large-english
Key                        | Status   |                                                                                         
---------------------------+----------+-----------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])            
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 1024]) vs model:torch.Size([3, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Step,Training Loss
10,1.176850
20,1.079368
30,1.063346
40,1.093342
50,1.151899
60,1.065312
70,1.179491
80,1.076885
90,1.076781
100,0.971235


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=598, training_loss=1.0796057468273967, metrics={'train_runtime': 617.76, 'train_samples_per_second': 7.741, 'train_steps_per_second': 0.968, 'total_flos': 1114127709249024.0, 'train_loss': 1.0796057468273967, 'epoch': 2.0})

In [16]:
trainer_RoBERTa.evaluate(test_dataset)

Training Loss,Validation Loss,Step
1.114964,1.046172,598


{'eval_loss': 1.0461715459823608}

In [17]:
y_pred = np.argmax(trainer_RoBERTa.predict(test_dataset).predictions, axis=-1)

In [18]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.49      1.00      0.66       293
           1       0.00      0.00      0.00       159
           2       0.00      0.00      0.00       146

    accuracy                           0.49       598
   macro avg       0.16      0.33      0.22       598
weighted avg       0.24      0.49      0.32       598



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


###BERT

In [19]:
from transformers import  AutoTokenizer

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [20]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=256)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=256)

In [21]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

In [22]:
from transformers import AutoModelForSequenceClassification,Trainer,TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10,
)

In [23]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

trainer_BERT = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer_BERT.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss
10,1.795697
20,1.329798
30,1.135096
40,1.101767
50,1.114629
60,1.043526
70,1.195904
80,1.078154
90,1.007597
100,0.986049


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=598, training_loss=0.899301619832731, metrics={'train_runtime': 265.6513, 'train_samples_per_second': 18.001, 'train_steps_per_second': 2.251, 'total_flos': 629115478631424.0, 'train_loss': 0.899301619832731, 'epoch': 2.0})

In [26]:
trainer_BERT.evaluate(test_dataset)

Training Loss,Validation Loss,Step
0.776251,0.910206,598


{'eval_loss': 0.9102055430412292}

In [28]:
y_pred = np.argmax(trainer_BERT.predict(test_dataset).predictions, axis=-1)

In [29]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.71      0.75      0.73       293
           1       0.57      0.56      0.56       159
           2       0.44      0.40      0.42       146

    accuracy                           0.61       598
   macro avg       0.57      0.57      0.57       598
weighted avg       0.61      0.61      0.61       598



###cardiffnlp/twitter-xlm-roberta

In [24]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-xlm-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [30]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [31]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

In [32]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10
)

In [33]:
from transformers import AutoModelForSequenceClassification, Trainer


model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(device)

trainer_XLMRoBERTa = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer_XLMRoBERTa.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Step,Training Loss
10,1.625623
20,1.284004
30,1.107147
40,1.079200
50,1.076611
60,1.026035
70,1.132416
80,1.037496
90,0.992830
100,1.034906


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=598, training_loss=0.8626835569490159, metrics={'train_runtime': 446.6518, 'train_samples_per_second': 10.706, 'train_steps_per_second': 1.339, 'total_flos': 314552090893824.0, 'train_loss': 0.8626835569490159, 'epoch': 2.0})

In [34]:
trainer_XLMRoBERTa.evaluate(test_dataset)

Training Loss,Validation Loss,Step
0.675057,0.909004,598


{'eval_loss': 0.909003734588623}

In [35]:
y_pred = np.argmax(trainer_XLMRoBERTa.predict(test_dataset).predictions, axis=-1)

In [36]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.69      0.76      0.73       293
           1       0.60      0.51      0.55       159
           2       0.43      0.41      0.42       146

    accuracy                           0.61       598
   macro avg       0.57      0.56      0.57       598
weighted avg       0.60      0.61      0.61       598



### DistilBERT

In [37]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [38]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [39]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

In [40]:
from transformers import (
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10,
)

In [41]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english", num_labels=3, ignore_mismatched_sizes=True).to(device)

trainer_distilBERT = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer_distilBERT.train()

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Step,Training Loss
10,1.061266
20,1.055131
30,1.073817
40,1.077763
50,1.067053
60,1.031613
70,1.139366
80,1.019462
90,0.981449
100,0.925892


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=598, training_loss=0.8509343595409075, metrics={'train_runtime': 118.3651, 'train_samples_per_second': 40.4, 'train_steps_per_second': 5.052, 'total_flos': 158367599304192.0, 'train_loss': 0.8509343595409075, 'epoch': 2.0})

In [42]:
trainer_distilBERT.evaluate(test_dataset)

Training Loss,Validation Loss,Step
0.751573,0.946209,598


{'eval_loss': 0.9462091326713562}

In [43]:
y_pred = np.argmax(trainer_distilBERT.predict(test_dataset).predictions, axis=-1)

In [44]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.69      0.74      0.71       293
           1       0.54      0.47      0.51       159
           2       0.37      0.36      0.36       146

    accuracy                           0.58       598
   macro avg       0.53      0.52      0.53       598
weighted avg       0.57      0.58      0.57       598

